In [ ]:
#| default_exp mcp

## MCP server

Expose nbskill notebook operations as native MCP tools. This is the preferred integration for careful single-notebook reads and edits because multiline notebook cells travel as structured tool arguments rather than shell-quoted strings. Keep MCP calls serial; use the CLI through uv run for batch operations and final verification.

The command-line functions are useful on their own, but coding agents work best when the same operations are available as structured tools. This notebook exposes the project through a FastMCP server while keeping the server layer thin and predictable.

The MCP server should stay boring on purpose. Each tool accepts structured arguments, captures printed output, uses notebook locks where file operations can collide, and delegates the actual work to the same functions tested elsewhere.

```python
mcp = create_mcp()
# MCP clients see tools such as read_nb, write_nb, update_cell, exec_nb, and diff_nb.
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import nbskill.mcp as _mcp_mod
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.mcp import capture_call as _example_capture_call
from nbskill.mcp import create_mcp as _example_create_mcp
from nbskill.read import read_nb as _example_read_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
def _demo_tool():
    print("captured output")

print(_example_capture_call(_demo_tool))
print(type(_example_create_mcp()).__name__)

captured output
FastMCP


In [ ]:
#| export
import os,json
import sys
from contextlib import redirect_stdout, redirect_stderr
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path

from fastcore.script import Param, call_parse
from fastmcp import FastMCP
from fastmcp.tools import ToolResult
from mcp.types import TextContent

from nbskill.convert import py2nb as _py2nb
from nbskill.convert import py2nbs as _py2nbs
from nbskill.edit_interactive import execute_plan as _execute_plan
from nbskill.execute import exec_nb as _exec_nb
from nbskill.graph import private_symbol_report as _private_symbol_report
from nbskill.graph import symbol_graph as _symbol_graph
from nbskill.parallel import notebook_locks
from nbskill.read import read_nb as _read_nb
from nbskill.read import show_doc as _show_doc
from nbskill.review import style_check as _style_check
from nbskill.review import diff_nb as _diff_nb
from nbskill.write import update_cell as _update_cell
from nbskill.write import write_nb as _write_nb


### Capturing command output

The MCP tools should return text, not leak stdout and stderr into the server process. These helpers capture each underlying function call and convert its visible result into one response string.

In [ ]:
#| export
def as_text(value):
    return "" if value is None else str(value)


def _package_version(name="nbskill"):
    try: return version(name)
    except PackageNotFoundError: return "unknown"


def capture_call(func, **kwargs):
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(as_text(result))
    return chr(10).join(chunk for chunk in chunks if chunk)


def capture_notebook_call(func, *paths, **kwargs):
    "Capture a call while holding per-notebook locks for `paths`."
    with notebook_locks(*paths):
        return capture_call(func, **kwargs)


def _json_preview(value, limit=1200):
    text = json.dumps(value, indent=2, sort_keys=True, default=str)
    if len(text) <= limit: return text
    return f"{text[:limit].rstrip()}\n... truncated ..."


def mcp_tool_result(tool, arguments, full_output):
    "Return visible MCP text plus structured data for clients that inspect it."
    call = {"tool": tool, "arguments": arguments}
    summary = "\n".join([
        f"{tool} completed",
        "",
        "Call:",
        _json_preview(call),
        "",
        "Result:",
        full_output or "",
    ])
    return ToolResult(
        content=[TextContent(type="text", text=summary)],
        structured_content={"summary": summary, "call": call, "full_output": full_output},
    )

### Registering notebook tools

`create_mcp` is the bridge between this package and an agent client. Each tool is a thin wrapper around a public function, with notebook locks around operations that touch shared files.

In [ ]:
#| export
def create_mcp():
    "Create the nbskill FastMCP server."
    capabilities = (
        "read_nb,show_doc,write_nb,update_cell,exec_nb,diff_nb,execute_plan,"
        "symbol_graph,private_symbol_report,style_check,py2nb,py2nbs"
    )
    mcp = FastMCP(
        "nbskill",
        instructions=(
            "Work notebook-first in nbdev projects. Prefer read_nb/show_doc for context, "
            "write_nb/update_cell for edits, exec_nb for safe visible notebook execution, "
            "and execute_plan for Lisette-powered single-notebook plan execution. "
            "Notebook operations are concurrency-safe: calls touching the same notebook are serialized, "
            "calls touching different notebooks can run in parallel, and execution uses a global semaphore. "
            "Keep documentation before exported code and show-off examples after it."
        ),
    )

    @mcp.tool(name="healthcheck")
    def healthcheck_tool() -> ToolResult:
        "Return a small status report for the local nbskill MCP server."
        full_output = "\n".join([
            "nbskill mcp ok",
            f"version={_package_version()}",
            f"cwd={Path.cwd()}",
            f"python={sys.executable}",
            f"pid={os.getpid()}",
            f"capabilities={capabilities}",
            "parallel=same-notebook operations serialized; different notebooks may run in parallel",
            "execution=global semaphore with one active safe notebook execution",
            "schema_refresh=restart or reconnect the MCP client after reinstall/export to refresh tool schemas",
        ])
        return mcp_tool_result("healthcheck", {}, full_output)

    @mcp.tool(name="read_nb")
    def read_nb_tool(
        path: str,
        query: str | None = None,
        cell_id: str | None = None,
        chapter: str | None = None,
        cell_type: str | None = None,
        contains: str | None = None,
        context: str = "overview",
        show_ids: bool = False,
    ) -> ToolResult:
        "Read compact notebook views; context controls overview, precise source, or full surrounding docs/examples."
        arguments = dict(
            path=path, query=query, cell_id=cell_id, chapter=chapter,
            cell_type=cell_type, contains=contains, context=context, show_ids=show_ids,
        )
        full_output = capture_notebook_call(
            _read_nb, path, path=path, query=query, cell_id=cell_id, chapter=chapter,
            cell_type=cell_type, contains=contains, context=context, show_ids=show_ids,
        )
        return mcp_tool_result("read_nb", arguments, full_output)

    @mcp.tool(name="show_doc")
    def show_doc_tool(path: str, symbol: str, context: int = 2, source: bool = False, show_ids: bool = False) -> ToolResult:
        "Show rationale/docs, exported code, and show-off examples for a symbol."
        arguments = dict(path=path, symbol=symbol, context=context, source=source, show_ids=show_ids)
        full_output = capture_notebook_call(_show_doc, path, path=path, symbol=symbol, context=context, source=source, show_ids=show_ids)
        return mcp_tool_result("show_doc", arguments, full_output)

    @mcp.tool(name="write_nb")
    def write_nb_tool(
        path: str,
        cells: str = "",
        before_id: str | None = None,
        after_id: str | None = None,
        chapter: str | None = None,
        replace: bool = False,
        cell_type: str = "code",
        export: bool = True,
        run_test: bool = False,
        run_style: bool = False,
        style_strict: bool = False,
        validate_code: bool = True,
        old_str: str | None = None,
        new_str: str | None = None,
        dry_run: bool = False,
        show_cells: bool = False,
    ) -> ToolResult:
        "Write cells to a notebook, or replace literal text across notebooks."
        arguments = dict(
            path=path, cells=cells, before_id=before_id, after_id=after_id,
            chapter=chapter, replace=replace, cell_type=cell_type, export=export,
            run_test=run_test, run_style=run_style, style_strict=style_strict,
            validate_code=validate_code, old_str=old_str, new_str=new_str,
            dry_run=dry_run, show_cells=show_cells,
        )
        full_output = capture_notebook_call(
            _write_nb, path, path=path, cells=cells, cells_file=None, before_id=before_id, after_id=after_id,
            chapter=chapter, replace=replace, cell_type=cell_type, export=export, run_test=run_test,
            run_style=run_style, style_strict=style_strict, validate_code=validate_code,
            old_str=old_str, new_str=new_str, dry_run=dry_run, show_cells=show_cells,
        )
        return mcp_tool_result("write_nb", arguments, full_output)

    @mcp.tool(name="update_cell")
    def update_cell_tool(
        path: str,
        new: str = "",
        cell_id: str | None = None,
        old_str: str | None = None,
        line_range: str | None = None,
        source_hash: str | None = None,
        cell_type: str = "code",
        export: bool = True,
        run_test: bool = False,
        validate_code: bool = True,
        dry_run: bool = False,
    ) -> ToolResult:
        "Update a cell, replace old_str, or replace 1-based inclusive line_range."
        arguments = dict(
            path=path, new=new, cell_id=cell_id, old_str=old_str,
            line_range=line_range, source_hash=source_hash, cell_type=cell_type,
            export=export, run_test=run_test, validate_code=validate_code,
            dry_run=dry_run,
        )
        full_output = capture_notebook_call(
            _update_cell, path, path=path, new=new, new_file=None, cell_id=cell_id, old_str=old_str,
            line_range=line_range, source_hash=source_hash, cell_type=cell_type, export=export,
            run_test=run_test, validate_code=validate_code, dry_run=dry_run,
        )
        return mcp_tool_result("update_cell", arguments, full_output)

    @mcp.tool(name="exec_nb")
    def exec_nb_tool(
        path: str,
        dest: str | None = None,
        exc_stop: bool = False,
        up2id: int | str | None = None,
        chapter: str | None = None,
        timeout: int = 30,
        show_output: bool = True,
        verbose: bool = False,
        safe: bool = True,
        allow: str | None = None,
        ok_dests: str | None = None,
        cache_httpx: bool = False,
        cache_dir: str | None = None,
        cache_domains: str | None = None,
        allow_new: bool = False,
    ) -> ToolResult:
        "Execute a notebook and return visible outputs/errors, safe by default."
        arguments = dict(
            path=path, dest=dest, exc_stop=exc_stop, up2id=up2id,
            chapter=chapter, timeout=timeout, show_output=show_output,
            verbose=verbose, safe=safe, allow=allow, ok_dests=ok_dests,
            cache_httpx=cache_httpx, cache_dir=cache_dir,
            cache_domains=cache_domains, allow_new=allow_new,
        )
        full_output = capture_notebook_call(
            _exec_nb, path, dest or path, path=path, dest=dest, exc_stop=exc_stop, up2id=up2id,
            chapter=chapter, timeout=timeout, show_output=show_output, verbose=verbose,
            safe=safe, allow=allow, ok_dests=ok_dests, cache_httpx=cache_httpx,
            cache_dir=cache_dir, cache_domains=cache_domains, allow_new=allow_new,
        )
        return mcp_tool_result("exec_nb", arguments, full_output)

    @mcp.tool(name="diff_nb")
    def diff_nb_tool(path: str, ref_a: str | None = "HEAD", ref_b: str | None = None, adds: bool = True, changes: bool = True, dels: bool = False) -> ToolResult:
        "Diff code cells only. nbskill metadata-only changes are summarized, not expanded."
        arguments = dict(path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels)
        full_output = capture_notebook_call(_diff_nb, path, path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels)
        return mcp_tool_result("diff_nb", arguments, full_output)

    @mcp.tool(name="execute_plan")
    def execute_plan_tool(
        notebook: str,
        plan: str,
        model: str | None = None,
        max_steps: int = 20,
        timeout: int = 30,
        export: bool = True,
    ) -> ToolResult:
        "Run a Lisette edit-interactive loop against one notebook."
        arguments = dict(notebook=notebook, plan=plan, model=model, max_steps=max_steps, timeout=timeout, export=export)
        full_output = capture_call(
            _execute_plan, notebook=notebook, plan=plan, model=model,
            max_steps=max_steps, timeout=timeout, export=export,
        )
        return mcp_tool_result("execute_plan", arguments, full_output)


    @mcp.tool(name="symbol_graph")
    def symbol_graph_tool(path: str = "nbs", symbol: str = "") -> ToolResult:
        "Show definitions, callers, and callees for a notebook symbol."
        arguments = dict(path=path, symbol=symbol)
        full_output = capture_call(_symbol_graph, path=path, symbol=symbol)
        return mcp_tool_result("symbol_graph", arguments, full_output)

    @mcp.tool(name="private_symbol_report")
    def private_symbol_report_tool(path: str = "nbs") -> ToolResult:
        "Show cross-notebook calls to private `_` symbols."
        arguments = dict(path=path)
        full_output = capture_call(_private_symbol_report, path=path)
        return mcp_tool_result("private_symbol_report", arguments, full_output)

    @mcp.tool(name="style_check")
    def style_check_tool(
        path: str = ".",
        skip_folder_re: str | None = None,
        skip_path: str | None = None,
        strict: bool = False,
        delete_after_output: bool = False,
        delete_after_outout: bool = False,
    ) -> ToolResult:
        "Print fast.ai style hints, notebook hygiene warnings, and global tool usage."
        arguments = dict(
            path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict,
            delete_after_output=delete_after_output, delete_after_outout=delete_after_outout,
        )
        full_output = capture_call(_style_check, **arguments)
        return mcp_tool_result("style_check", arguments, full_output)

    @mcp.tool(name="py2nb")
    def py2nb_tool(path: str, nbs_path: str = "nbs", dest: str | None = None, class_lines: int = 100, method_lines: int = 10) -> ToolResult:
        "Convert a Python file into an nbdev notebook."
        arguments = dict(path=path, nbs_path=nbs_path, dest=dest, class_lines=class_lines, method_lines=method_lines)
        full_output = capture_call(_py2nb, path=path, nbs_path=nbs_path, dest=dest, class_lines=class_lines, method_lines=method_lines)
        return mcp_tool_result("py2nb", arguments, full_output)

    @mcp.tool(name="py2nbs")
    def py2nbs_tool(path: str, nbs_path: str = "nbs", recursive: bool = True, maxdepth: int | None = None, preserve_tree: bool = True, class_lines: int = 100, method_lines: int = 10) -> ToolResult:
        "Convert Python files in a folder into nbdev notebooks."
        arguments = dict(
            path=path, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
            preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines,
        )
        full_output = capture_call(
            _py2nbs, path=path, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
            preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines,
        )
        return mcp_tool_result("py2nbs", arguments, full_output)

    return mcp

### Running the server

The CLI entry point only chooses the transport and starts FastMCP. Keeping startup separate from tool registration makes `create_mcp` easy to test without launching a long-running server.

In [ ]:
#| export
@call_parse
def main(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    create_mcp().run(transport=transport, show_banner=show_banner)

In [ ]:
mcp = create_mcp()
tools = {tool.name: tool for tool in await mcp.list_tools()}
assert {"healthcheck", "read_nb", "write_nb", "update_cell", "exec_nb", "show_doc", "execute_plan", "symbol_graph", "private_symbol_report"} <= set(tools)
assert "show_cells" in str(tools["write_nb"].parameters)
health = await mcp.call_tool("healthcheck", {})
health_text = str(health)
assert "version=" in health_text
assert "capabilities=" in health_text
assert "schema_refresh=" in health_text

[05/18/26 20:15:24] Error calling tool 'healthcheck'                                                               
                    ╭───────────────────────────── Traceback (most recent call last) ─────────────────────────────╮
                    │ /Users/macbook/Projects/nbskill/.venv/lib/python3.12/site-packages/fastmcp/server/server.py │
                    │ :1274 in call_tool                                                                          │
                    │                                                                                             │
                    │ /Users/macbook/Projects/nbskill/.venv/lib/python3.12/site-packages/fastmcp/tools/base.py:37 │
                    │ 9 in _run                                                                                   │
                    │                                                                                             │
                    │                                   ... 7 frames hidden ...                                   │
                    │                                                                                             │
                    │ in mcp_tool_result:41                                                                       │
                    │                                                                                             │
                    │   38 │   │   f"{tool} completed",                                                           │
                    │   39 │   │   "",                                                                            │
                    │   40 │   │   "Call:",                                                                       │
                    │ ❱ 41 │   │   _json_preview(call),                                                           │
                    │   42 │   │   "",                                                                            │
                    │   43 │   │   "Result:",                                                                     │
                    │   44 │   │   full_output or "",                                                             │
                    │                                                                                             │
                    │ in _json_preview:29                                                                         │
                    │                                                                                             │
                    │   26                                                                                        │
                    │   27                                                                                        │
                    │   28 def _json_preview(value, limit=1200):                                                  │
                    │ ❱ 29 │   text = json.dumps(value, indent=2, sort_keys=True, default=str)                    │
                    │   30 │   if len(text) <= limit: return text                                                 │
                    │   31 │   return f"{text[:limit].rstrip()}\n... truncated ..."                               │
                    │   32                                                                                        │
                    ╰─────────────────────────────────────────────────────────────────────────────────────────────╯
                    NameError: name 'json' is not defined

ToolError: Error calling tool 'healthcheck': name 'json' is not defined

In [ ]:
calls = {}
old_execute_plan = _mcp_mod._execute_plan

try:
    def fake_execute_plan(**kwargs):
        calls.update(kwargs)
        return "delegated"

    _mcp_mod._execute_plan = fake_execute_plan
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool(
        "execute_plan",
        {
            "notebook": "nbs/index.ipynb",
            "plan": "noop",
            "model": "fake",
            "max_steps": 1,
            "timeout": 2,
            "export": False,
        },
    )
    assert calls == {
        "notebook": "nbs/index.ipynb",
        "plan": "noop",
        "model": "fake",
        "max_steps": 1,
        "timeout": 2,
        "export": False,
    }
    assert "delegated" in str(result)
finally:
    _mcp_mod._execute_plan = old_execute_plan

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File ~/Projects/nbskill/.venv/lib/python3.12/site-packages/fastmcp/server/server.py:1274, in FastMCP.call_tool(self, name, arguments, version, run_middleware, task_meta)
   1273 try:
-> 1274     return await tool._run(arguments or {}, task_meta=task_meta)
   1275 except FastMCPError as e:

File ~/Projects/nbskill/.venv/lib/python3.12/site-packages/fastmcp/tools/base.py:379, in Tool._run(self, arguments, task_meta)
    377     return task_result
--> 379 return await self.run(arguments)

File ~/Projects/nbskill/.venv/lib/python3.12/site-packages/fastmcp/tools/function_tool.py:331, in FunctionTool.run(self, arguments)
    330 elif self.run_in_thread:
--> 331     result = await call_sync_fn_in_threadpool(
    332         type_adapter.validate_python, arguments
    333     )
    334     if inspect.isawaitable(result):

File ~/Projects/nbskil

[05/18/26 20:00:41] Error calling tool 'execute_plan'                           
                    ╭─────────── Traceback (most recent call last) ────────────╮
                    │ /Users/macbook/Projects/nbskill/.venv/lib/python3.12/sit │
                    │ e-packages/fastmcp/server/server.py:1274 in call_tool    │
                    │                                                          │
                    │ /Users/macbook/Projects/nbskill/.venv/lib/python3.12/sit │
                    │ e-packages/fastmcp/tools/base.py:379 in _run             │
                    │                                                          │
                    │                 ... 7 frames hidden ...                  │
                    │                                                          │
                    │ /Users/macbook/Projects/nbskill/nbskill/mcp.py:74 in     │
                    │ mcp_tool_result                                          │
                    │       

ToolError: Error calling tool 'execute_plan': name 'json' is not defined

In [ ]:
path = demo_path("07_mcp_sample.ipynb")
_write_nb(path,new_nb([mk_cell("#| default_exp sample", cell_type="code")]))
text = capture_call(_read_nb, path=str(path), context="overview")
assert "default_exp sample" in text
remove_demo_path(path)

Wrote 1 cells to nbs/data/07_mcp_sample.ipynb and exported with nbdev


Path('nbs/data/07_mcp_sample.ipynb')

In [ ]:
assert as_text(None) == ""
assert as_text({"ok": True}) == "{'ok': True}"
assert capture_call(lambda: "returned") == "returned"

def _prints_and_returns():
    print("printed")
    return "returned"

assert capture_call(_prints_and_returns) == "printed"